In [ ]:
from pathlib import Path
import sys

def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "src").exists() and (candidate / "datasets").exists():
            return candidate
    raise RuntimeError("Repository root not found.")

REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

FIELD_OPS_RAW_DIR = REPO_ROOT / "datasets" / "field_operations" / "raw"
FIELD_OPS_ARCHIVE_DIR = REPO_ROOT / "archive" / "datasets" / "field_operations" / "old_datasets_files"
FIELD_OPS_ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)


In [5]:
# %% FIX: Control Location Sharing — corrigir span longo do slot location_sharing_action
# - Encontra automaticamente onde está a lista de exemplos no JSON (qualquer estrutura)
# - Divide "enable location sharing" -> entity só em "enable"
# - O restante (" location sharing") é acoplado no INÍCIO do próximo segmento de texto (se existir)

import json, re
from pathlib import Path
from collections import Counter

# ========= CONFIG =========
DATASET_PATH = FIELD_OPS_RAW_DIR / "control_location_sharing_300.json"
if not DATASET_PATH.exists():
    raise FileNotFoundError(DATASET_PATH)

TARGET_ENTITY = "location_sharing_action"
OUT_PATH = FIELD_OPS_ARCHIVE_DIR / f"{DATASET_PATH.stem}_fixed.json"

# ========= LOAD =========
with open(DATASET_PATH, "r", encoding="utf-8") as f:
    dataset = json.load(f)

# ========= FIND EXAMPLES LIST (robusto) =========
def is_example_list(x):
    return (
        isinstance(x, list) and len(x) > 0 and
        all(isinstance(it, dict) and isinstance(it.get("data"), list) for it in x)
    )

def get_by_path(obj, path):
    cur = obj
    for p in path:
        cur = cur[p]
    return cur

def find_example_lists(obj):
    candidates = []

    def rec(o, path=()):
        if is_example_list(o):
            candidates.append(path)
            # não retorna aqui; pode haver listas aninhadas também

        if isinstance(o, dict):
            for k, v in o.items():
                rec(v, path + (k,))
        elif isinstance(o, list):
            for i, v in enumerate(o):
                rec(v, path + (i,))

    rec(obj, ())
    return candidates

if isinstance(dataset, list) and is_example_list(dataset):
    examples = dataset
    examples_path = ("<root_list>",)
else:
    paths = find_example_lists(dataset)
    if not paths:
        # debug mínimo pra você enxergar a estrutura
        print("Top-level keys:", list(dataset.keys()) if isinstance(dataset, dict) else type(dataset))
        raise ValueError("Não encontrei nenhuma lista de exemplos (lista de dicts com chave 'data').")

    # escolhe a maior lista candidata
    best_path = max(paths, key=lambda p: len(get_by_path(dataset, p)))
    examples = get_by_path(dataset, best_path)
    examples_path = best_path

print("[OK] Lista de exemplos encontrada em:", examples_path)
print("[OK] Total de exemplos:", len(examples))

# ========= SPLIT RULES =========
action_counter = Counter()
changed_examples = 0
changed_spans = 0

def split_action_and_tail(span_text: str):
    """
    Retorna (leading_ws, action, tail).
    - Dá suporte a 'check in' e 'check-in' como ação composta.
    """
    if span_text is None:
        return None

    # Caso especial: "check in ..." / "check-in ..."
    m = re.match(r"^(\s*)(check\s+in|check-in)\b(.*)$", span_text, flags=re.IGNORECASE)
    if m:
        leading_ws = m.group(1)
        action = "check-in"  # normaliza
        tail = m.group(3)
        return leading_ws, action, tail

    # Caso geral: primeira "palavra" (inclui hífen)
    m = re.match(r"^(\s*)([\w-]+)(.*)$", span_text)
    if not m:
        return None

    leading_ws, action, tail = m.group(1), m.group(2), m.group(3)
    return leading_ws, action, tail

# ========= APPLY FIX =========
for ex in examples:
    if not isinstance(ex, dict) or "data" not in ex or not isinstance(ex["data"], list):
        continue

    segs = ex["data"]
    new_segs = []
    example_changed = False

    i = 0
    while i < len(segs):
        seg = segs[i]
        seg = dict(seg)  # cópia defensiva

        if seg.get("entity") != TARGET_ENTITY:
            new_segs.append(seg)
            i += 1
            continue

        original_text = seg.get("text", "")
        parts = split_action_and_tail(original_text)

        # se não der match, mantém
        if not parts:
            new_segs.append(seg)
            i += 1
            continue

        leading_ws, action, tail = parts

        # conta action
        action_counter[action.lower()] += 1

        # se já é só 1 token (tail vazio/whitespace), não muda
        if tail.strip() == "":
            new_segs.append(seg)
            i += 1
            continue

        # move leading_ws pro segmento anterior se existir (pra não poluir o slot)
        if leading_ws:
            if new_segs:
                new_segs[-1]["text"] = new_segs[-1].get("text", "") + leading_ws
            else:
                action = leading_ws + action  # raro, mas mantém íntegro

        # 1) action como entity
        new_segs.append({"text": action, "entity": TARGET_ENTITY})

        # 2) tail deve ser acoplado no início do PRÓXIMO texto (se houver e se for texto puro)
        if tail:
            if i + 1 < len(segs):
                nxt = dict(segs[i + 1])
                nxt_ent = nxt.get("entity", None)
                if "text" in nxt and (nxt_ent is None or nxt_ent == "" or "entity" not in nxt):
                    nxt["text"] = tail + nxt.get("text", "")
                    segs[i + 1] = nxt  # atualiza o próximo item (será processado normalmente)
                else:
                    # próximo é entity (ou não tem text): cria um novo segmento de texto
                    new_segs.append({"text": tail})
            else:
                # não existe próximo: cria um novo segmento de texto
                new_segs.append({"text": tail})

        example_changed = True
        changed_spans += 1
        i += 1  # avança (o próximo já foi alterado se aplicável)

    if example_changed:
        ex["data"] = new_segs
        changed_examples += 1

# ========= SAVE =========
with open(OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(dataset, f, ensure_ascii=False, indent=2)

# ========= REPORT =========
print("\n=== FIX REPORT ===")
print("Arquivo lido:  ", DATASET_PATH)
print("Arquivo salvo: ", OUT_PATH)
print("Exemplos alterados:", changed_examples)
print("Spans corrigidos:  ", changed_spans)

print("\nActions encontradas (top 50):")
for a, c in action_counter.most_common(50):
    print(f"  {a:>15}  ->  {c}")


[OK] Lista de exemplos encontrada em: ('Control Location Sharing',)
[OK] Total de exemplos: 300

=== FIX REPORT ===
Arquivo lido:   E:\My Intent Recognition Project\generated_datasets\control_location_sharing_300.json
Arquivo salvo:  E:\My Intent Recognition Project\generated_datasets\control_location_sharing_300_fixed.json
Exemplos alterados: 300
Spans corrigidos:   300

Actions encontradas (top 50):
             turn  ->  41
           enable  ->  34
            pause  ->  33
             stop  ->  28
         activate  ->  27
           resume  ->  26
            start  ->  23
         check-in  ->  23
       deactivate  ->  23
            share  ->  20
          disable  ->  17
              end  ->  4
            begin  ->  1


In [6]:
# %% FIX: mover "the" do início do location_reference para o final do segmento anterior

import json, re
from pathlib import Path

INPUT_PATH  = FIELD_OPS_RAW_DIR / "incident_status_300.json"
OUTPUT_PATH = FIELD_OPS_ARCHIVE_DIR / f"{INPUT_PATH.stem}_fix_the_location_reference.json"

TARGET_ENTITY = "location"
WRITE_OUTPUT = True

def iter_data_lists(obj, path=""):
    if isinstance(obj, dict):
        for k, v in obj.items():
            new_path = f"{path}.{k}" if path else k
            if k == "data" and isinstance(v, list):
                yield new_path, v
            else:
                yield from iter_data_lists(v, new_path)
    elif isinstance(obj, list):
        for i, v in enumerate(obj):
            yield from iter_data_lists(v, f"{path}[{i}]")

def move_the_prefix(prev_text: str, entity_text: str):
    """
    Se entity_text começa com 'the ' (ou 'The '), remove esse prefixo e anexa ao final de prev_text.
    Retorna (new_prev_text, new_entity_text) ou (None, None) se não houver mudança.
    """
    m = re.match(r"^(the)(\s+)(.+)$", entity_text, flags=re.IGNORECASE)
    if not m:
        return None, None

    the_word = m.group(1)   # preserva 'the' ou 'The' conforme o texto original
    rest     = m.group(3)   # restante do texto após "the" + espaços

    # normaliza o espaço: garante 1 espaço antes de anexar "the "
    prev = prev_text.rstrip()
    if prev != "":
        prev += " "
    prev += the_word.lower() + " "   # força 'the' minúsculo ao mover (melhor no meio da frase)

    new_ent = rest.lstrip()

    return prev, new_ent

with INPUT_PATH.open("r", encoding="utf-8") as f:
    payload = json.load(f)

changes = []

for data_path, data_list in iter_data_lists(payload):
    for i, cur in enumerate(data_list):
        if not isinstance(cur, dict):
            continue
        if cur.get("entity") != TARGET_ENTITY:
            continue

        cur_text = cur.get("text")
        if not isinstance(cur_text, str):
            continue
        if i == 0:
            continue

        prev = data_list[i - 1]
        if not isinstance(prev, dict) or not isinstance(prev.get("text"), str):
            continue

        old_prev_text = prev["text"]
        old_cur_text  = cur_text

        new_prev_text, new_cur_text = move_the_prefix(old_prev_text, old_cur_text)
        if new_prev_text is None:
            continue

        prev["text"] = new_prev_text
        cur["text"]  = new_cur_text

        changes.append({
            "where_prev": f"{data_path}[{i-1}].text",
            "where_cur":  f"{data_path}[{i}].text",
            "old_prev": old_prev_text,
            "new_prev": new_prev_text,
            "old_cur": old_cur_text,
            "new_cur": new_cur_text,
        })

print(f"Total de ocorrências corrigidas: {len(changes)}\n")
for c in changes:
    print(f"- PREV @ {c['where_prev']}")
    print(f"  OLD: {repr(c['old_prev'])}")
    print(f"  NEW: {repr(c['new_prev'])}")
    print(f"  CUR  @ {c['where_cur']}")
    print(f"  OLD: {repr(c['old_cur'])}")
    print(f"  NEW: {repr(c['new_cur'])}\n")

if WRITE_OUTPUT:
    with OUTPUT_PATH.open("w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)
    print(f"Arquivo salvo em: {OUTPUT_PATH}")


Total de ocorrências corrigidas: 35

- PREV @ Incident Status[100].data[0].text
  OLD: 'Firefighters have established a secure perimeter around '
  NEW: 'Firefighters have established a secure perimeter around the '
  CUR  @ Incident Status[100].data[1].text
  OLD: 'the warehouse district'
  NEW: 'warehouse district'

- PREV @ Incident Status[103].data[0].text
  OLD: 'No significant changes reported at '
  NEW: 'No significant changes reported at the '
  CUR  @ Incident Status[103].data[1].text
  OLD: 'the downtown plaza'
  NEW: 'downtown plaza'

- PREV @ Incident Status[106].data[0].text
  OLD: 'Authorities are maintaining control at '
  NEW: 'Authorities are maintaining control at the '
  CUR  @ Incident Status[106].data[1].text
  OLD: 'the industrial complex'
  NEW: 'industrial complex'

- PREV @ Incident Status[107].data[0].text
  OLD: 'Response teams remain stationed at '
  NEW: 'Response teams remain stationed at the '
  CUR  @ Incident Status[107].data[1].text
  OLD: 'the city h

In [9]:
# %% FIX: casualty_count -> manter só a 1ª palavra (ex: "three") e mover o resto (ex: "victims") para o próximo text

import json
import json, re
from pathlib import Path

INPUT_PATH  = FIELD_OPS_RAW_DIR / "medical_evacuation_300.json"
OUTPUT_PATH = FIELD_OPS_ARCHIVE_DIR / f"{INPUT_PATH.stem}_fix_casualty_count.json"

TARGET_ENTITY = "casualty_count"
WRITE_OUTPUT = True

def iter_data_lists(obj, path=""):
    """Percorre recursivamente o JSON e retorna (path, data_list) para cada chave 'data' que seja lista."""
    if isinstance(obj, dict):
        for k, v in obj.items():
            new_path = f"{path}.{k}" if path else k
            if k == "data" and isinstance(v, list):
                yield new_path, v
            else:
                yield from iter_data_lists(v, new_path)
    elif isinstance(obj, list):
        for i, v in enumerate(obj):
            yield from iter_data_lists(v, f"{path}[{i}]")

def split_first_token(text: str):
    """
    Divide em:
      - first: primeiro token não-espaço (ex: 'three' ou '3')
      - rest: tudo após o primeiro token, começando pelo primeiro espaço (ex: ' victims ...')
    Retorna (first, rest) ou (None, None) se não houver o que mover.
    """
    if not isinstance(text, str):
        return None, None
    m = re.match(r"^\s*(\S+)(\s+.+)$", text)
    if not m:
        return None, None
    first = m.group(1)
    rest  = m.group(2)  # inclui o(s) espaço(s) antes da 2ª palavra (ex: " victims")
    return first, rest

with INPUT_PATH.open("r", encoding="utf-8") as f:
    payload = json.load(f)

changes = []

for data_path, data_list in iter_data_lists(payload):
    for i, cur in enumerate(data_list):
        if not isinstance(cur, dict):
            continue
        if cur.get("entity") != TARGET_ENTITY:
            continue
        if not isinstance(cur.get("text"), str):
            continue

        first, rest = split_first_token(cur["text"])
        if first is None:
            continue  # nada para mover (ex: já é só "three")

        old_cur = cur["text"]

        # garante próximo chunk para receber o "rest"
        if i + 1 < len(data_list) and isinstance(data_list[i + 1], dict) and isinstance(data_list[i + 1].get("text"), str):
            nxt = data_list[i + 1]
            old_next = nxt["text"]

            cur["text"] = first
            nxt["text"] = rest + old_next

            changes.append({
                "where_cur":  f"{data_path}[{i}].text",
                "where_next": f"{data_path}[{i+1}].text",
                "old_cur": old_cur,
                "new_cur": cur["text"],
                "old_next": old_next,
                "new_next": nxt["text"],
            })
        else:
            # não existe próximo: cria um chunk plain no final
            cur["text"] = first
            data_list.insert(i + 1, {"text": rest})

            changes.append({
                "where_cur":  f"{data_path}[{i}].text",
                "where_next": f"{data_path}[{i+1}].text (inserted)",
                "old_cur": old_cur,
                "new_cur": cur["text"],
                "old_next": None,
                "new_next": rest,
            })

print(f"Total de ocorrências corrigidas: {len(changes)}\n")
for c in changes:
    print(f"- CUR  @ {c['where_cur']}")
    print(f"  OLD: {repr(c['old_cur'])}")
    print(f"  NEW: {repr(c['new_cur'])}")
    print(f"  NEXT @ {c['where_next']}")
    print(f"  OLD: {repr(c['old_next'])}")
    print(f"  NEW: {repr(c['new_next'])}\n")

if WRITE_OUTPUT:
    with OUTPUT_PATH.open("w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)
    print(f"Arquivo salvo em: {OUTPUT_PATH}")


Total de ocorrências corrigidas: 225

- CUR  @ Medical Evacuation[0].data[3].text
  OLD: 'three people'
  NEW: 'three'
  NEXT @ Medical Evacuation[0].data[4].text
  OLD: ' at the site.'
  NEW: ' people at the site.'

- CUR  @ Medical Evacuation[1].data[3].text
  OLD: 'five injured'
  NEW: 'five'
  NEXT @ Medical Evacuation[1].data[4].text
  OLD: ' from the mountain.'
  NEW: ' injured from the mountain.'

- CUR  @ Medical Evacuation[2].data[3].text
  OLD: 'two casualties'
  NEW: 'two'
  NEXT @ Medical Evacuation[2].data[4].text
  OLD: ' stranded on the riverbank.'
  NEW: ' casualties stranded on the riverbank.'

- CUR  @ Medical Evacuation[3].data[3].text
  OLD: 'four victims'
  NEW: 'four'
  NEXT @ Medical Evacuation[3].data[4].text
  OLD: ' at the accident scene.'
  NEW: ' victims at the accident scene.'

- CUR  @ Medical Evacuation[4].data[3].text
  OLD: 'six people'
  NEW: 'six'
  NEXT @ Medical Evacuation[4].data[4].text
  OLD: ' trapped on the roof?'
  NEW: ' people trapped on the

In [10]:
# %% FIX: stream_control_action -> manter só a 1ª palavra e mover o resto (com espaço) p/ o próximo "text"

import json, re
from pathlib import Path

INPUT_PATH  = FIELD_OPS_RAW_DIR / "control_live_video_stream_300.json"  # ajuste se necess?rio
OUTPUT_PATH = FIELD_OPS_ARCHIVE_DIR / f"{INPUT_PATH.stem}_fix_stream_control_action.json"

TARGET_ENTITY = "stream_control_action"
WRITE_OUTPUT = True

def iter_data_lists(obj, path=""):
    """Percorre recursivamente o JSON e retorna (path, data_list) para cada chave 'data' que seja lista."""
    if isinstance(obj, dict):
        for k, v in obj.items():
            new_path = f"{path}.{k}" if path else k
            if k == "data" and isinstance(v, list):
                yield new_path, v
            else:
                yield from iter_data_lists(v, new_path)
    elif isinstance(obj, list):
        for i, v in enumerate(obj):
            yield from iter_data_lists(v, f"{path}[{i}]")

def split_first_word_keep_rest_with_space(text: str):
    """
    Ex: 'stop the live stream' -> ('stop', ' the live stream')
    Retorna (first, rest_with_leading_space) ou (None, None) se não houver o que mover.
    """
    if not isinstance(text, str):
        return None, None
    m = re.match(r"^\s*(\S+)(\s+.+)$", text)
    if not m:
        return None, None
    first = m.group(1)
    rest  = m.group(2)  # já inclui o(s) espaços antes do restante (ex: ' the live stream')
    return first, rest

with INPUT_PATH.open("r", encoding="utf-8") as f:
    payload = json.load(f)

changes = []

for data_path, data_list in iter_data_lists(payload):
    for i, cur in enumerate(data_list):
        if not isinstance(cur, dict):
            continue
        if cur.get("entity") != TARGET_ENTITY:
            continue
        if not isinstance(cur.get("text"), str):
            continue

        first, rest = split_first_word_keep_rest_with_space(cur["text"])
        if first is None:
            continue  # já é só uma palavra (ex: 'stop')

        old_cur = cur["text"]

        # envia o rest para o próximo chunk (preferencialmente plain)
        if i + 1 < len(data_list) and isinstance(data_list[i + 1], dict) and isinstance(data_list[i + 1].get("text"), str):
            nxt = data_list[i + 1]
            old_next = nxt["text"]

            cur["text"] = first
            nxt["text"] = rest + old_next

            changes.append({
                "where_cur":  f"{data_path}[{i}].text",
                "where_next": f"{data_path}[{i+1}].text",
                "old_cur": old_cur,
                "new_cur": cur["text"],
                "old_next": old_next,
                "new_next": nxt["text"],
            })
        else:
            # não existe próximo: cria um chunk plain no final
            cur["text"] = first
            data_list.insert(i + 1, {"text": rest})

            changes.append({
                "where_cur":  f"{data_path}[{i}].text",
                "where_next": f"{data_path}[{i+1}].text (inserted)",
                "old_cur": old_cur,
                "new_cur": cur["text"],
                "old_next": None,
                "new_next": rest,
            })

print(f"Total de ocorrências corrigidas: {len(changes)}\n")
for c in changes:
    print(f"- CUR  @ {c['where_cur']}")
    print(f"  OLD: {repr(c['old_cur'])}")
    print(f"  NEW: {repr(c['new_cur'])}")
    print(f"  NEXT @ {c['where_next']}")
    print(f"  OLD: {repr(c['old_next'])}")
    print(f"  NEW: {repr(c['new_next'])}\n")

if WRITE_OUTPUT:
    with OUTPUT_PATH.open("w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)
    print(f"Arquivo salvo em: {OUTPUT_PATH}")


Total de ocorrências corrigidas: 300

- CUR  @ Control Live Video Stream[0].data[1].text
  OLD: 'start streaming'
  NEW: 'start'
  NEXT @ Control Live Video Stream[0].data[2].text
  OLD: ' the live video now.'
  NEW: ' streaming the live video now.'

- CUR  @ Control Live Video Stream[1].data[1].text
  OLD: 'stop the live feed'
  NEW: 'stop'
  NEXT @ Control Live Video Stream[1].data[2].text
  OLD: ' immediately?'
  NEW: ' the live feed immediately?'

- CUR  @ Control Live Video Stream[2].data[1].text
  OLD: 'begin broadcasting'
  NEW: 'begin'
  NEXT @ Control Live Video Stream[2].data[2].text
  OLD: ' the video from the drone.'
  NEW: ' broadcasting the video from the drone.'

- CUR  @ Control Live Video Stream[3].data[1].text
  OLD: 'end the live stream'
  NEW: 'end'
  NEXT @ Control Live Video Stream[3].data[2].text
  OLD: ' for this operation.'
  NEW: ' the live stream for this operation.'

- CUR  @ Control Live Video Stream[4].data[1].text
  OLD: 'activate the live stream'
  NEW: 

In [ ]:
# %% FIX: mover "the" do início do location_reference para o final do segmento anterior

import json, re
from pathlib import Path

INPUT_PATH  = FIELD_OPS_RAW_DIR / "access_external_cameras_300.json"
OUTPUT_PATH = FIELD_OPS_ARCHIVE_DIR / f"{INPUT_PATH.stem}_fix_the_camera_search_area.json"

TARGET_ENTITY = "camera_search_area"
WRITE_OUTPUT = True

def iter_data_lists(obj, path=""):
    if isinstance(obj, dict):
        for k, v in obj.items():
            new_path = f"{path}.{k}" if path else k
            if k == "data" and isinstance(v, list):
                yield new_path, v
            else:
                yield from iter_data_lists(v, new_path)
    elif isinstance(obj, list):
        for i, v in enumerate(obj):
            yield from iter_data_lists(v, f"{path}[{i}]")

def move_the_prefix(prev_text: str, entity_text: str):
    """
    Se entity_text começa com 'the ' (ou 'The '), remove esse prefixo e anexa ao final de prev_text.
    Retorna (new_prev_text, new_entity_text) ou (None, None) se não houver mudança.
    """
    m = re.match(r"^(the)(\s+)(.+)$", entity_text, flags=re.IGNORECASE)
    if not m:
        return None, None

    the_word = m.group(1)   # preserva 'the' ou 'The' conforme o texto original
    rest     = m.group(3)   # restante do texto após "the" + espaços

    # normaliza o espaço: garante 1 espaço antes de anexar "the "
    prev = prev_text.rstrip()
    if prev != "":
        prev += " "
    prev += the_word.lower() + " "   # força 'the' minúsculo ao mover (melhor no meio da frase)

    new_ent = rest.lstrip()

    return prev, new_ent

with INPUT_PATH.open("r", encoding="utf-8") as f:
    payload = json.load(f)

changes = []

for data_path, data_list in iter_data_lists(payload):
    for i, cur in enumerate(data_list):
        if not isinstance(cur, dict):
            continue
        if cur.get("entity") != TARGET_ENTITY:
            continue

        cur_text = cur.get("text")
        if not isinstance(cur_text, str):
            continue
        if i == 0:
            continue

        prev = data_list[i - 1]
        if not isinstance(prev, dict) or not isinstance(prev.get("text"), str):
            continue

        old_prev_text = prev["text"]
        old_cur_text  = cur_text

        new_prev_text, new_cur_text = move_the_prefix(old_prev_text, old_cur_text)
        if new_prev_text is None:
            continue

        prev["text"] = new_prev_text
        cur["text"]  = new_cur_text

        changes.append({
            "where_prev": f"{data_path}[{i-1}].text",
            "where_cur":  f"{data_path}[{i}].text",
            "old_prev": old_prev_text,
            "new_prev": new_prev_text,
            "old_cur": old_cur_text,
            "new_cur": new_cur_text,
        })

print(f"Total de ocorrências corrigidas: {len(changes)}\n")
for c in changes:
    print(f"- PREV @ {c['where_prev']}")
    print(f"  OLD: {repr(c['old_prev'])}")
    print(f"  NEW: {repr(c['new_prev'])}")
    print(f"  CUR  @ {c['where_cur']}")
    print(f"  OLD: {repr(c['old_cur'])}")
    print(f"  NEW: {repr(c['new_cur'])}\n")

if WRITE_OUTPUT:
    with OUTPUT_PATH.open("w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)
    print(f"Arquivo salvo em: {OUTPUT_PATH}")
